# PEDE — Generator Figur Bibliometrik / SLNA (Google Colab)

Menghasilkan figur *science-mapping* (thematic map, keyword co-occurrence, annual production, top sources, collaboration network) **secara TER-SKRIP & DETERMINISTIK** langsung dari **MongoDB SLR** — pengganti VOSviewer/biblioshiny manual.

Output: PNG/SVG/PDF **+ data CSV mentah + manifest.json** → simpan bersama deposit **Zenodo** untuk reproducibility (menutup rantai dokumentasi). Baca-saja; kredensial dari **Colab Secrets**.

> Ringan — **tidak butuh GPU**. Runtime CPU cukup.


## 1. (Opsional) Mount Google Drive
Untuk menyimpan figur ke Drive. Lewati bila cukup unduh dari Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Setup Kode (Clone / Git Pull)
Ambil kode terbaru `ifcoid/PEDE` (logika inti ada di `core/biblio_figures.py`, notebook hanya orkestrasi).


In [ ]:
import os
REPO_URL = "https://github.com/ifcoid/PEDE.git"
WORK_DIR = "/content/PEDE"
if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull
os.chdir(WORK_DIR)
print('CWD:', os.getcwd())


## 3. Instalasi Dependensi
Ringan (tanpa GPU): `pymongo` (baca korpus) + `matplotlib`/`networkx` (figur).


In [ ]:
!pip install -q pymongo matplotlib networkx python-dotenv


## 4. Konfigurasi (Secrets & Sesi)
Tambahkan **🔑 Secrets** Colab (sidebar kiri), aktifkan *Notebook access*:
- `MONGO_URI` — connection string MongoDB Atlas/lokal (read-only disarankan)
- `DB_NAME` *(opsional)* — default `slr_agentic_db`
- `TELEGRAM_BOT_TOKEN` / `TELEGRAM_CHAT_ID` *(opsional)* — notifikasi selesai

Lalu isi **`SESSION_ID`** (dari `slr_sessions._id` — lihat di UI/laporan).


In [ ]:
from google.colab import userdata
def _secret(name):
    try: return (userdata.get(name) or '').strip()
    except Exception: return ''
os.environ['MONGO_URI'] = _secret('MONGO_URI')
os.environ['DB_NAME'] = _secret('DB_NAME') or 'slr_agentic_db'

# === ISI session_id Anda di sini ===
SESSION_ID = 'slr_contoh'  #@param {type:'string'}
MIN_KEYWORD_FREQ = 2       #@param {type:'integer'}
USE_ALL_SCREENING = False  #@param {type:'boolean'}

# Simpan ke Drive bila ter-mount, else lokal Colab.
OUTDIR = f"/content/drive/MyDrive/slr_figures/{SESSION_ID}" if os.path.isdir('/content/drive/MyDrive') else f"/content/PEDE/data/figures/{SESSION_ID}"
assert os.environ['MONGO_URI'], 'MONGO_URI belum diset di Colab Secrets!'
print('Output:', OUTDIR)


## 5. Generate Figur
Deterministik: run ulang menghasilkan figur identik. Menulis PNG/SVG/PDF + CSV + `manifest.json`.


In [ ]:
from core.biblio_figures import generate_all
manifest = generate_all(SESSION_ID, OUTDIR, min_freq=MIN_KEYWORD_FREQ, use_all=USE_ALL_SCREENING)
import json; print(json.dumps(manifest, indent=2, ensure_ascii=False))


## 6. Tampilkan Figur


In [ ]:
from IPython.display import Image, display
for fig in manifest.get('figures', []):
    if fig.get('files'):
        png = [f for f in fig['files'] if f.endswith('.png')]
        if png:
            print('==', fig['figure'], '==')
            display(Image(filename=os.path.join(OUTDIR, png[0])))
    elif fig.get('skipped'):
        print('(dilewati)', fig['figure'], '->', fig['skipped'])


## 7. (Opsional) Notifikasi Telegram + Ringkasan Artefak
Artefak (figur + CSV + manifest) siap di-unggah ke **Zenodo** bersama protokol & paket reproducibility. Sitasi DOI-nya di manuskrip.


In [ ]:
import urllib.parse, urllib.request
files = sorted(os.listdir(OUTDIR))
print(f"{len(files)} artefak di {OUTDIR}:"); [print(' -', f) for f in files]
tok, chat = _secret('TELEGRAM_BOT_TOKEN'), _secret('TELEGRAM_CHAT_ID')
if tok and chat:
    n = sum(1 for f in manifest.get('figures', []) if f.get('files'))
    msg = f"\U0001F4CA Figur bibliometrik SLR selesai: {n} figur, korpus {manifest.get('corpus_size',0)} studi (sesi {SESSION_ID}). Artefak+CSV siap untuk Zenodo."
    url = f"https://api.telegram.org/bot{tok}/sendMessage?" + urllib.parse.urlencode({'chat_id': chat, 'text': msg})
    try: urllib.request.urlopen(url, timeout=15); print('Notif terkirim.')
    except Exception as e: print('Notif gagal:', e)
